> Bu, Tom Augspurger'ın CC-BY-4.0 lisanslı 'Effective Pandas' eserinin gönüllü Türkçe çevirisidir. Orijinal: https://github.com/TomAugspurger/effective-pandas

# Effective Pandas

## Giriş (Introduction)

Bu seri, Python programlama dili için bir veri analizi kütüphanesi olan [pandas](http://pandas.pydata.org)'ın nasıl etkili biçimde kullanılacağını ele alır.
Orta düzey bir kitleyi hedefler: pandas ile biraz deneyimi olan, ancak kendini geliştirmek isteyen kişiler için hazırlanmıştır.

## Önceki Çalışmalar (Prior Art)

pandas öğrenmek için pek çok değerli kaynak bulunmaktadır; bu seri bunlardan biri değildir.
Yeni başlayanlar için, özellikle SQL'e aşina olanlara, genellikle [Greg Reda'nın](https://twitter.com/gjreda) [3 bölümlük giriş yazısı](http://gregreda.com/2013/10/26/intro-to-pandas-data-structures/) önerilir. Elbette pandas'ın kendi [dokümantasyonu](http://pandas.pydata.org/) da mevcuttur. Video formatını tercih edenler için PyData Seattle'da giriş niteliğinde [bir konuşma](https://www.youtube.com/watch?v=otCriSKVV_8) yapılmıştır. Wes McKinney'in [Python for Data Analysis](http://shop.oreilly.com/product/0636920023784.do) kitabı hâlâ başvurulacak temel kaynaktır (aynı zamanda NumPy'a da gerçekten iyi bir giriştir). Jake VanderPlas'ın erken erişimde olan [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) kitabı da oldukça başarılıdır.
Kevin Markham'ın, pandas öğrenmeye yeni başlayanlar için hazırladığı bir [video serisi](http://www.dataschool.io/easier-data-analysis-with-pandas/) de bulunmaktadır.

Bunca kaynağın yanında (ve gözden kaçırılarak haksızlık edilen daha nicesinin) bir yenisi neden yazılsın? Artık azalan verim yasasının devreye girmesi kaçınılmaz gibi görünüyor.
Yine de Mart 2016 itibarıyla güncel olan ve deyimsel, yani pandas'a özgü (*pandorable*) kod yazımını vurgulayan bir rehbere yer olduğu düşünülmüştür. <!-- TR-NOT: "pandorable" (pandas + Pythonic esintili bir uydurma sözcük) için tam karşılık Türkçede yok; anlamı açıklanarak orijinal terim italik korunmuştur. -->
Bu seri, Python'a veya NumPy ve pandas'a tamamen yeni başlayanlar için muhtemelen uygun olmayacaktır.
Bir tesadüf eseri, bu ilk yazı görece giriş niteliğinde konuları kapsamaktadır; dolayısıyla bağlantısı verilen kaynaklardan bazıları okunup geri dönülebilir, ya da soru olması durumunda [yazara ulaşılabilir](https://twitter.com/tomaugspurger).

## Verinin Edinilmesi (Get the Data)

BTS'ten alınan [uçuş gecikme verileriyle](http://www.transtats.bts.gov/databases.asp?Mode_ID=1&Mode_Desc=Aviation&Subject_ID2=0) çalışılacaktır (R kullanıcıları benzer veriler için Hadley'nin [NYCFlights13](https://github.com/hadley/nycflights13) veri kümesini kurabilir.


In [1]:
import os
import zipfile

import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if int(os.environ.get("MODERN_PANDAS_EPUB", 0)):
    import prep

In [2]:
import requests

headers = {
    'Referer': 'https://www.transtats.bts.gov/DL_SelectFields.asp?Table_ID=236&DB_Short_Name=On-Time',
    'Origin': 'https://www.transtats.bts.gov',
    'Content-Type': 'application/x-www-form-urlencoded',
}

params = (
    ('Table_ID', '236'),
    ('Has_Group', '3'),
    ('Is_Zipped', '0'),
)

with open('modern-1-url.txt', encoding='utf-8') as f:
    data = f.read().strip()

os.makedirs('data', exist_ok=True)
dest = "data/flights.csv.zip"

if not os.path.exists(dest):
    r = requests.post('https://www.transtats.bts.gov/DownLoad_Table.asp',
                      headers=headers, params=params, data=data, stream=True)

    with open("data/flights.csv.zip", 'wb') as f:
        for chunk in r.iter_content(chunk_size=102400): 
            if chunk:
                f.write(chunk)

Bu indirme işlemi bir ZIP dosyası döndürmüştür.
Tek bir CSV içeren ZIP arşivlerinin otomatik olarak açılmasını sağlayacak açık bir [Pull Request](https://github.com/pydata/pandas/pull/12175) bulunmaktadır; ancak şimdilik dosyanın elle çıkarılıp öyle okunması gerekmektedir.

In [3]:
zf = zipfile.ZipFile("data/flights.csv.zip")
fp = zf.extract(zf.filelist[0].filename, path='data/')
df = pd.read_csv(fp, parse_dates=["FL_DATE"]).rename(columns=str.lower)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450017 entries, 0 to 450016
Data columns (total 33 columns):
fl_date                  450017 non-null datetime64[ns]
unique_carrier           450017 non-null object
airline_id               450017 non-null int64
tail_num                 449378 non-null object
fl_num                   450017 non-null int64
origin_airport_id        450017 non-null int64
origin_airport_seq_id    450017 non-null int64
origin_city_market_id    450017 non-null int64
origin                   450017 non-null object
origin_city_name         450017 non-null object
dest_airport_id          450017 non-null int64
dest_airport_seq_id      450017 non-null int64
dest_city_market_id      450017 non-null int64
dest                     450017 non-null object
dest_city_name           450017 non-null object
crs_dep_time             450017 non-null int64
dep_time                 441476 non-null float64
dep_delay                441476 non-null float64
taxi_out                

## İndeksleme (Indexing)

Ya da, *açık olmak örtük olmaktan iyidir* (explicit is better than implicit).
Yapılan sayıma göre, [Stackoverflow](http://stackoverflow.com/questions/tagged/pandas?sort=votes&pageSize=15)'daki en çok oy alan ilk 15 pandas sorusundan 7'si indeksleme hakkındadır. Başlamak için burası uygun bir nokta gibi görünüyor.

İndeksleme ile kastedilen, bir DataFrame veya Series'in alt kümelerinin seçilmesidir.
`DataFrame`ler (ve daha az ölçüde `Series`ler) zorlu bir dizi güçlükle karşılaşılmasına yol açar:

- Listelerde olduğu gibi konuma göre indeksleme yapılabilir.
- Sözlüklerde olduğu gibi etikete göre indeksleme yapılabilir.
- NumPy dizilerinde olduğu gibi boolean maskelerle indeksleme yapılabilir.
- Bu indeksleyicilerin her biri skaler bir indeks, bir dizi ya da bir `slice` olabilir.
- Bunların her biri bir DataFrame'in indeksi (satır etiketleri) veya sütunları üzerinde çalışabilmelidir.
- Ve bunların her biri hiyerarşik indeksler üzerinde de çalışabilmelidir.

pandas'ın indekslemesindeki karmaşıklık, genel olarak pandas API'sinin karmaşıklığının küçük bir örneğidir.
Bu karmaşıklığın bir nedeni vardır (en azından çoğunun), ancak bu, öğrenme sürecindeyken pek de teselli olmuyor.
Yine de tüm bu indeksleme yöntemleri, kütüphanede yer almalarını haklı çıkaracak kadar gerçekten kullanışlıdır.

## Dilimleme (Slicing)

Ya da, *açık olmak örtük olmaktan iyidir* (explicit is better than implicit).

Yapılan sayıma göre, [Stackoverflow](http://stackoverflow.com/questions/tagged/pandas?sort=votes&pageSize=15)'daki en çok oy alan ilk 15 pandas sorusundan 7'si dilimleme hakkındadır. Başlamak için burası uygun bir nokta gibi görünüyor.

Kısa bir tarihsel not: Yıllarca satır ve/veya sütun seçimi için tercih edilen yöntem `.ix` olmuştur.

In [5]:
df.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
10,2014-01-01,N3LGAA
11,2014-01-01,N368AA
12,2014-01-01,N3DDAA
13,2014-01-01,N332AA
14,2014-01-01,N327AA
15,2014-01-01,N3LBAA


Ancak bu basit küçük işlem, bazı karmaşıklıkları gizlemektedir. Varsayılan `range(n)` indeksi yerine, aşağıdaki gibi bir tam sayı indeksi olsaydı ne olurdu?

In [6]:
first = df.groupby('airline_id')[['fl_date', 'unique_carrier']].first()
first.head()

,fl_date,unique_carrier
airline_id,,
19393,2014-01-01,WN
19690,2014-01-01,HA
19790,2014-01-01,DL
19805,2014-01-01,AA
19930,2014-01-01,AS


Yukarıdaki dilimin `.ix`'e verildiğinde ne döndüreceği önceden tahmin edilebilir mi?

In [7]:
first.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
airline_id,,


Sürpriz, boş bir DataFrame! Bu da veri analizinde nadiren iyi bir şeydir. Ne oldu?

Tam sayı indeks kullanıldığından, `.ix` çağrısı etiket tabanlı modunu kullanmıştır. 10 ile 15 arasındaki (dahil) tam sayı *etiketleri* aranmıştır. Hiçbiri bulunamamıştır. Bir aralık dilimlendiği için KeyError yükseltmek yerine boş bir DataFrame döndürülmüştür.

Buna karşılık, tam sayılar yerine bir string indeks olduğu varsayılsın.

In [8]:
first = df.groupby('unique_carrier').first()
first.ix[10:15, ['fl_date', 'tail_num']]

,fl_date,tail_num
unique_carrier,,
UA,2014-01-01,N14214
US,2014-01-01,N650AW
VX,2014-01-01,N637VA
WN,2014-01-01,N412WN



Ve yine çalışıyor! Bu sefer bir string indeks olduğundan, `.ix` konumsal modunu kullanmıştır. 10-15 arasındaki *satırlar* aranmıştır (sağ sınır hariç).

Ancak dilimin sonucu güvenilir biçimde önceden tahmin edilemez. `.ix`'in etiket tabanlı indeksleme mi (boş DataFrame döndüren) yoksa konumsal indeksleme mi (son örnekteki gibi) kullanacağını kestirebilmek için veri tiplerinin (dtype) bilinmesi, kodun *okuyucusuna* (muhtemelen gelecekteki kendisine) düşer.
Genel olarak, `.ix`'in tam sayı indekslerde etiket tabanlı, tam sayı olmayanlarda ise konum tabanlı indekslemeye yönlendirilmesinde olduğu gibi, davranışı veriye bağlı olan metotların doğru kullanılması zordur. Bu tür metotlar pandas'tan ayıklanmaya çalışılmaktadır.

pandas 0.12'den bu yana bu görevler net biçimde iki metoda ayrılmıştır:

1. Etiket tabanlı indeksleme için `.loc`
2. Konumsal indeksleme için `.iloc`

In [9]:
first.loc[['AA', 'AS', 'DL'], ['fl_date', 'tail_num']]

,fl_date,tail_num
unique_carrier,,
AA,2014-01-01,N338AA
AS,2014-01-01,N524AS
DL,2014-01-01,N911DL


In [10]:
first.iloc[[0, 1, 3], [0, 1]]

,fl_date,airline_id
unique_carrier,,
AA,2014-01-01,19805
AS,2014-01-01,19930
DL,2014-01-01,19790


`.ix` hâlâ mevcuttur ve yakın zamanda kullanımdan kaldırılması söz konusu değildir. Zaman zaman kullanışlı olabilir. Ancak `.ix` alışkanlıktan ya da başka bir seçenek bilinmediğinden kullanılıyorsa, `.loc` ve `.iloc` denenebilir. Meraklı okuyucular için, pandas'ın temel geliştiricilerinden Joris Van den Bossche, pandas `__getitem__` API'sine dair [kapsamlı bir derleme](https://github.com/pydata/pandas/issues/9595) hazırlamıştır.
Bu serideki sonraki bir yazı, indekslerin etkili kullanımını daha ayrıntılı ele alacaktır;
indeksler kendi başlarına da kullanışlı nesnelerdir, ancak şimdilik yakından ilişkili bir konuya geçilecektir.

## SettingWithCopy

pandas, atamaların çalışmıyormuş gibi görünmesiyle ilgili eskiden *çok sayıda* soru almaktaydı. Temsili bir örnek olarak [şu StackOverflow](http://stackoverflow.com/q/16553298/1889400) sorusu ele alınacaktır.

In [11]:
f = pd.DataFrame({'a':[1,2,3,4,5], 'b':[10,20,30,40,50]})
f

,a,b
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


Kullanıcı, `a` değeri 3 veya daha küçük olan satırlardaki `b` değerlerini alıp bunları `b / 10`'a eşitlemek istemiştir.
Bu satırları seçmek için `f['a'] <= 3` ile boolean indeksleme kullanılacaktır,

In [12]:
# ignore the context manager for now
with pd.option_context('mode.chained_assignment', None):
    f[f['a'] <= 3]['b'] = f[f['a'] <= 3 ]['b'] / 10
f

,a,b
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


Ve hiçbir şey olmamıştır. Aslında bir şey olmuştur, ama kimse buna tanık olmamıştır. Hiçbir referansı olmayan bir nesne değiştirilirse, bu bir ses çıkarır mı?

Yukarıda context manager ile susturulan uyarı, oldukça yardımcı olan [bir açıklamaya](http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy) bağlantı vermektedir. Ana noktalar burada özetlenecektir.

`f`'in güncellenememesi, kaçınılması gereken bir uygulama olan *zincirleme indeksleme* (chained indexing) olarak adlandırılan durumdan kaynaklanmaktadır.
"Zincirleme" ifadesi, tek bir indeksleme işlemi yerine art arda birden fazla kez indeksleme yapılmasından gelmektedir.
Yukarıda eşitliğin sol tarafında iki işlem gerçekleştirilmiştir: biri `__getitem__`, diğeri `__setitem__` (python'da köşeli parantezler, atama için kullanıldığında `__getitem__` veya `__setitem__` için sözdizimsel kolaylıktır). Böylece `f[f['a'] <= 3]['b']` şu hale gelir:

1. `getitem`: `f[f['a'] <= 3]`
2. `setitem`: `_['b'] = ...`  # `_`, 1. adımın sonucunu temsil etmek için kullanılmıştır.

Genel olarak pandas, bu ilk `__getitem__`'in altta yatan verinin bir view'ini mi yoksa bir kopyasını mı döndürdüğünü garanti edemez. <!-- TR-NOT: "view" terimi sözlükte yer almıyor; pandas/numpy bağlamında yaygın kullanımıyla İngilizce bırakıldı. -->
Değişiklikler, yukarıda `_` olarak adlandırılan, yani 1. adımdaki `__getitem__`'in sonucu olan nesne üzerinde *yapılacaktır*.
Ancak `_`'in orijinal `f` ile aynı belleği paylaşıp paylaşmadığı bilinmemektedir.
Dolayısıyla `_` üzerinde yapılan değişikliklerin `f`'e yansıyıp yansımayacağından emin olunamaz.

Doğru biçimde yapılması gerekirse, şöyle yazılmalıdır:

In [13]:
f.loc[f['a'] <= 3, 'b'] = f.loc[f['a'] <= 3, 'b'] / 10
f

,a,b
0,1,1.0
1,2,2.0
2,3,3.0
3,4,40.0
4,5,50.0


Artık her şey `__setitem__`'e yapılan tek bir çağrıda gerçekleşmekte ve pandas atamanın doğru biçimde yapılmasını garanti edebilmektedir.

Genel kural şudur: art arda gelen köşeli parantezler, yani `][`, görüldüğünde sorun yaşanabilir. Bunun yerine bir `.loc[..., ...]` kullanmak sorunu ortadan kaldırır.

Diğer bir öneri de şudur: bir SettingWithCopy uyarısı, *atama* yapıldığı anda yükseltilir.
Olası kopya ise kodun daha önceki bir noktasında oluşmuş olabilir.

## Çok Boyutlu İndeksleme (Multidimensional Indexing)

MultiIndex'ler pandas'ın belki de en sevilen özelliğidir.
Daha yüksek boyutlu veri kümelerinin, zihnin bazen daha kolay kavrayabildiği alışılagelmiş iki boyutlu bir tabloda gösterilmesini sağlarlar.
MultiIndex'in her ek düzeyi (level) başka bir boyutu temsil eder.
Bunun bedeli ise biraz daha zor bir etiket indekslemesidir.

pandas'a yapılan ilk hata bildirimi, [Kasım 2012](https://github.com/pydata/pandas/issues/2207)'de, bir MultiIndex'e indeksleme yapmakla ilgiliydi.
Bu konu şimdi gündeme getirilmektedir, çünkü elde edilen sonucun gerçekten bir hata olup olmadığı o zaman anlaşılamamıştır.
Ayrıca o hata bildiriminden bir alıntı:

> Bu gerçekten bir hata değilse özür dilerim. Python'da hâlâ çok yeniyim. Teşekkürler!

Ne sevimli.

Bu işlem, 2014 yılında yapılan ve bir MultiIndex'in herhangi bir düzeyinin dilimlenmesine olanak tanıyan [bu eklemeyle](http://pandas.pydata.org/pandas-docs/version/0.18.0/whatsnew.html#multiindexing-using-slicers) çok daha kolay hale gelmiştir.
Üzerinde çalışılacak bir MultiIndex'li DataFrame oluşturulsun.

In [14]:
hdf = df.set_index(['unique_carrier', 'origin', 'dest', 'tail_num', 'fl_date']).sort_index()
hdf[hdf.columns[:4]].head()

airline_id  fl_num  \
unique_carrier origin dest tail_num fl_date                          
AA             ABQ    DFW  N200AA   2014-01-06       19805    1662   
                                    2014-01-27       19805    1090   
                           N202AA   2014-01-27       19805    1332   
                           N426AA   2014-01-09       19805    1662   
                                    2014-01-15       19805    1467   

                                                origin_airport_id  \
unique_carrier origin dest tail_num fl_date                         
AA             ABQ    DFW  N200AA   2014-01-06              10140   
                                    2014-01-27              10140   
                           N202AA   2014-01-27              10140   
                           N426AA   2014-01-09              10140   
                                    2014-01-15              10140   

                                                origin_airport_seq_id  
unique_carrier origin dest tail_num fl_date                            
AA             ABQ    DFW  N200AA   2014-01-06                1014002  
                                    2014-01-27                1014002  
                           N202AA   2014-01-27                1014002  
                           N426AA   2014-01-09                1014002  
                                    2014-01-15                1014002

Terminolojiyi netleştirmek gerekirse, bir MultiIndex'in *düzeyleri* (levels), eski sütun adlarıdır
(`unique_carrier`, `origin`...).
Etiketler (labels) ise bir düzeydeki gerçek değerlerdir (`'AA'`, `'ABQ'`, ...).
Düzeylere isimleriyle veya konumlarıyla başvurulabilir; 0, en dıştaki düzeyi temsil eder.

En dıştaki indeks düzeyinin dilimlenmesi oldukça kolaydır; bunun için olağan `.loc[row_indexer, column_indexer]` kullanılması yeterlidir. Taşıyıcının American Airlines, Delta veya US Airways olduğu satırlardan `dep_time` ve `dep_delay` sütunları seçilecektir.

In [15]:
hdf.loc[['AA', 'DL', 'US'], ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             ABQ    DFW  N200AA   2014-01-06    1246.0       71.0
                                    2014-01-27     605.0        0.0
                           N202AA   2014-01-27     822.0      -13.0
                           N426AA   2014-01-09    1135.0        0.0
                                    2014-01-15    1022.0       -8.0
...                                                  ...        ...
US             TUS    PHX  N824AW   2014-01-16    1900.0      -10.0
                                    2014-01-20    1903.0       -7.0
                           N836AW   2014-01-08    1928.0       18.0
                                    2014-01-29    1908.0       -2.0
                           N837AW   2014-01-10    1902.0       -8.0

[139194 rows x 2 columns]

Şimdiye kadar sorun yok. Ya kalkış noktası Chicago O'Hare (`ORD`) veya Des Moines International Airport (DSM) olan satırların seçilmesi istenseydi?
`.loc`, `[row_indexer, column_indexer]` beklediğinden, satır indeksleyicisinin iki öğesi (taşıyıcı listesi ve kalkış noktası listesi) tek bir birim haline getirilmek üzere bir tuple içine alınabilir:

In [16]:
hdf.loc[(['AA', 'DL', 'US'], ['ORD', 'DSM']), ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             DSM    DFW  N200AA   2014-01-12     603.0       -7.0
                                    2014-01-17     751.0      101.0
                           N424AA   2014-01-10    1759.0       -1.0
                                    2014-01-15    1818.0       18.0
                           N426AA   2014-01-07    1835.0       35.0
...                                                  ...        ...
US             ORD    PHX  N806AW   2014-01-26    1406.0       -4.0
                           N830AW   2014-01-28    1401.0       -9.0
                           N833AW   2014-01-10    1500.0       50.0
                           N837AW   2014-01-19    1408.0       -2.0
                           N839AW   2014-01-14    1406.0       -4.0

[5205 rows x 2 columns]

Şimdi de yalnızca o taşıyıcılara değil, ORD veya DSM'den kalkan tüm uçuşlara bakılsın.
Bu eskiden oldukça sıkıntılı bir işlemdi.
`.xs` metoduna başvurmak, ya da `df.index.get_level_values(0)`'ı alıp istenen indeksleyicilerle zip'lemek, ya da belki indeksi sıfırlayıp bir boolean maske uygulayıp indeksi yeniden ayarlamak gerekebilirdi... çok can sıkıcı.

Ama artık bir `IndexSlice` kullanılabilir.

In [17]:
hdf.loc[pd.IndexSlice[:, ['ORD', 'DSM']], ['dep_time', 'dep_delay']]

dep_time  dep_delay
unique_carrier origin dest tail_num fl_date                        
AA             DSM    DFW  N200AA   2014-01-12     603.0       -7.0
                                    2014-01-17     751.0      101.0
                           N424AA   2014-01-10    1759.0       -1.0
                                    2014-01-15    1818.0       18.0
                           N426AA   2014-01-07    1835.0       35.0
...                                                  ...        ...
WN             DSM    MDW  N941WN   2014-01-17    1759.0       14.0
                           N943WN   2014-01-10    2229.0      284.0
                           N963WN   2014-01-22     656.0       -4.0
                           N967WN   2014-01-30     654.0       -6.0
                           N969WN   2014-01-19    1747.0        2.0

[22380 rows x 2 columns]

`:`, bu düzeydeki her etiketin dahil edileceğini belirtir.
`IndexSlice` nesnesi, her düzeyi dilimlemek için gereken gerçek python `slice` nesnesi için yalnızca bir sözdizimsel kolaylıktır. <!-- TR-NOT: Orijinal cümle ("needed to remove slice each level") kaynak metinde de dilbilgisi açısından tuhaf; anlaşılan kastına göre çevrilmiştir. -->

In [18]:
pd.IndexSlice[:, ['ORD', 'DSM']]

(slice(None, None, None), ['ORD', 'DSM'])

`hdf.loc[(:, ['ORD', 'DSM'])]` geçerli bir python sözdizimi olmadığından `IndexSlice` kullanılır.
Artık gönlünce dilimleme yapılabilir; Ocak ayının ilk yarısında O'Hare'den Des Moines'e giden tüm uçuşlar mı? Elbette, neden olmasın?

In [19]:
hdf.loc[pd.IndexSlice[:, 'ORD', 'DSM', :, '2014-01-01':'2014-01-15'],
        ['dep_time', 'dep_delay', 'arr_time', 'arr_delay']]

dep_time  dep_delay  arr_time  \
unique_carrier origin dest tail_num fl_date                                     
EV             ORD    DSM  NaN      2014-01-07       NaN        NaN       NaN   
                           N11121   2014-01-05       NaN        NaN       NaN   
                           N11181   2014-01-12    1514.0        6.0    1625.0   
                           N11536   2014-01-10    1723.0        4.0    1853.0   
                           N11539   2014-01-01    1127.0      127.0    1304.0   
...                                                  ...        ...       ...   
UA             ORD    DSM  N24212   2014-01-09    2023.0        8.0    2158.0   
                           N73256   2014-01-15    2019.0        4.0    2127.0   
                           N78285   2014-01-07    2020.0        5.0    2136.0   
                                    2014-01-13    2014.0       -1.0    2114.0   
                           N841UA   2014-01-11    1825.0       20.0    1939.0   

                                                arr_delay  
unique_carrier origin dest tail_num fl_date                
EV             ORD    DSM  NaN      2014-01-07        NaN  
                           N11121   2014-01-05        NaN  
                           N11181   2014-01-12       -2.0  
                           N11536   2014-01-10       19.0  
                           N11539   2014-01-01      149.0  
...                                                   ...  
UA             ORD    DSM  N24212   2014-01-09       34.0  
                           N73256   2014-01-15        3.0  
                           N78285   2014-01-07       12.0  
                                    2014-01-13      -10.0  
                           N841UA   2014-01-11       19.0  

[153 rows x 4 columns]

İndekslerle (MultiIndex'ler dahil) çalışma konusuna sonraki bir yazıda daha ayrıntılı değinilecektir. Kanıtlanmamış bir tez olarak, `IndexSlice`'ın yeterince kullanılmaması nedeniyle indekslerin de yeterince kullanılmadığı ve bunun da insanların indeksleri olduğundan daha hantal sanmasına yol açtığı düşünülmektedir. Ama birinci bölüm burada tamamlansın.

## Özet (WrapUp)

Bu ilk yazı, pandas'ın merkezinde yer alan bir konu olan indekslemeyi ele almıştır.
DataFrame'in sağladığı güç, bazı kaçınılmaz karmaşıklıklarla birlikte gelir.
En iyi uygulamalar (`.loc` ve `.iloc` kullanımı) pek çok baş ağrısından koruyacaktır.
Ardından sık yanlış anlaşılan iki alt konu, kopya üzerinde ayarlama yapma (SettingWithCopy) ve hiyerarşik indeksleme, ele alınmıştır.